In [1]:
import altair as alt
import pandas as pd
import geopandas as gpd # Requires geopandas -- e.g.: conda install -c conda-forge geopandas
alt.data_transformers.enable('json') # Let Altair/Vega-Lite work with large data sets

pass

In [3]:
names = pd.read_csv("Names_hints/dpt2020.csv", sep=";")
names.drop(names[names.preusuel == '_PRENOMS_RARES'].index, inplace=True)
names.drop(names[names.dpt == 'XX'].index, inplace=True)

names.sample(5)

,sexe,preusuel,annais,dpt,nombre
2389281,2,ERIKA,1984,14,3
2678051,2,JESSIE,1990,93,8
1742219,2,ABIGAELLE,2010,33,3
95933,1,ALLAIN,1938,56,6
852530,1,JEROME,1953,02,3


In [4]:
depts = gpd.read_file('Names_hints/departements-version-simplifiee.geojson')

depts.sample(5)

# Keep a reference around to the plain pandas dataframe, without geometry data, just in case
just_names = names

names = depts.merge(names, how='right', left_on='code', right_on='dpt')

names.sample(5)


grouped = (
    names.groupby(['dpt', 'preusuel', 'sexe'], as_index=False)
         .agg({'nombre': 'sum'})
)
grouped = depts.merge(grouped, how='right', left_on='code', right_on='dpt') # Add geometry data back in
grouped

,code,nom,geometry,dpt,preusuel,sexe,nombre
0,01,Ain,"POLYGON ((4.78021 46.17668, 4.79458 46.21832, ...",01,AARON,1,160
1,01,Ain,"POLYGON ((4.78021 46.17668, 4.79458 46.21832, ...",01,ABBY,2,3
2,01,Ain,"POLYGON ((4.78021 46.17668, 4.79458 46.21832, ...",01,ABDALLAH,1,7
3,01,Ain,"POLYGON ((4.78021 46.17668, 4.79458 46.21832, ...",01,ABDEL,1,3
4,01,Ain,"POLYGON ((4.78021 46.17668, 4.79458 46.21832, ...",01,ABDELKADER,1,3
...,...,...,...,...,...,...,...
239574,NaN,NaN,None,974,ÉSAÏE,1,3
239575,NaN,NaN,None,974,ÉTHAN,1,53
239576,NaN,NaN,None,974,ÉTIENNE,1,3
239577,NaN,NaN,None,974,ÉVA,2,32


In [7]:

# Top 10 prénoms les plus donnés
top_names = (
    names.groupby('preusuel')['nombre']
         .sum()
         .nlargest(10)
         .index
)

time_data = (
    names[names.preusuel.isin(top_names)]
    .groupby(['annais', 'preusuel'], as_index=False)
    ['nombre']
    .sum()
)

alt.Chart(time_data).mark_line().encode(
    x=alt.X('annais:O', title='Année'),
    y=alt.Y('nombre:Q', title='Naissances'),
    color='preusuel:N',
    tooltip=['annais', 'preusuel', 'nombre']
).properties(
    width=1200,
    height=600,
    title='Évolution des prénoms les plus populaires'
).interactive()

alt.Chart(...)

In [8]:
import altair as alt
import pandas as pd

# Top 10 prénoms les plus donnés
top_names = (
    names.groupby('preusuel')['nombre']
         .sum()
         .nlargest(10)
         .index
)

time_data = (
    names[names.preusuel.isin(top_names)]
    .groupby(['annais', 'preusuel'], as_index=False)['nombre']
    .sum()
)

# Calcul du pourcentage par année
totaux_annuels = (
    time_data.groupby('annais', as_index=False)['nombre']
             .sum()
             .rename(columns={'nombre': 'total_annuel'})
)

time_data = time_data.merge(totaux_annuels, on='annais')
time_data['pourcentage'] = (
    100 * time_data['nombre'] / time_data['total_annuel']
)

# Bouton radio Nombre / Pourcentage
mode = alt.param(
    name='Mode',
    value='Nombre',
    bind=alt.binding_radio(
        options=['Nombre', 'Pourcentage'],
        name='Affichage : '
    )
)

chart = (
    alt.Chart(time_data)
    .add_params(mode)
    .transform_calculate(
        valeur="""
        Mode == 'Nombre'
        ? datum.nombre
        : datum.pourcentage
        """
    )
    .mark_line()
    .encode(
        x=alt.X('annais:O', title='Année'),
        y=alt.Y(
            'valeur:Q',
            title='Naissances / Pourcentage'
        ),
        color=alt.Color('preusuel:N', title='Prénom'),
        tooltip=[
            alt.Tooltip('annais:O', title='Année'),
            alt.Tooltip('preusuel:N', title='Prénom'),
            alt.Tooltip('nombre:Q', title='Naissances'),
            alt.Tooltip('pourcentage:Q', title='Pourcentage', format='.2f')
        ]
    )
    .properties(
        width=1200,
        height=600,
        title='Évolution des prénoms les plus populaires'
    )
    .interactive()
)

chart

alt.Chart(...)

In [10]:
# Sélecteur de prénom
prenom = alt.param(
    name='Prenom',
    value='Tous',
    bind=alt.binding_select(
        options=['Tous'] + top_names,
        name='Prénom : '
    )
)

chart = (
    alt.Chart(time_data)
    .add_params(mode, prenom)
    .transform_filter(
        "(Prenom == 'Tous') || (datum.preusuel == Prenom)"
    )
    .transform_calculate(
        valeur="""
        Mode == 'Nombre'
        ? datum.nombre
        : datum.pourcentage
        """
    )
    .mark_line(point=True)
    .encode(
        x=alt.X('annais:O', title='Année'),
        y=alt.Y(
            'valeur:Q',
            title='Naissances / Pourcentage'
        ),
        color=alt.Color('preusuel:N', title='Prénom'),
        tooltip=[
            'annais:O',
            'preusuel:N',
            alt.Tooltip('nombre:Q', title='Naissances'),
            alt.Tooltip('pourcentage:Q', title='Pourcentage', format='.2f')
        ]
    )
    .properties(
        width=1200,
        height=600,
        title='Évolution des prénoms'
    )
    .interactive()
)

chart

alt.Chart(...)

In [24]:
import pandas as pd
import numpy as np
import altair as alt

# ==========================================================
# 1. Détection des prénoms à effet de mode
# ==========================================================

all_names = (
    names.groupby(["annais", "preusuel"], as_index=False)["nombre"]
         .sum()
)

scores = []

MIN_TOTAL = 3000

for prenom, g in all_names.groupby("preusuel"):

    total = g["nombre"].sum()

    if total < MIN_TOTAL:
        continue

    g = g.sort_values("annais").reset_index(drop=True)

    pos_pic = g["nombre"].idxmax()
    pic = g.loc[pos_pic, "nombre"]

    avant = g.loc[:pos_pic-1, "nombre"]
    apres = g.loc[pos_pic+1:, "nombre"]

    if len(avant) < 5 or len(apres) < 5:
        continue

    moyenne_avant = avant.mean()
    moyenne_apres = apres.mean()

    largeur_pic = (g["nombre"] >= 0.5 * pic).sum()

    score_mode = (
        (pic / (moyenne_avant + 1))
        * (pic / (moyenne_apres + 1))
        / largeur_pic
    )

    scores.append({
        "preusuel": prenom,
        "score_mode": score_mode,
        "pic": pic,
        "largeur_pic": largeur_pic,
        "total": total
    })

mode_scores = (
    pd.DataFrame(scores)
    .sort_values("score_mode", ascending=False)
    .reset_index(drop=True)
)

TOP_MODE = 10

prenoms_mode = (
    mode_scores
    .head(TOP_MODE)["preusuel"]
    .tolist()
)

print("Prénoms détectés comme effet de mode :")
print(prenoms_mode)

# ==========================================================
# 2. Top prénoms historiques
# ==========================================================

TOP_POP = 10

top_names = (
    names.groupby("preusuel")["nombre"]
         .sum()
         .nlargest(TOP_POP)
         .index
         .tolist()
)

# Union des deux ensembles
prenoms_affiches = sorted(
    set(top_names) | set(prenoms_mode)
)

# ==========================================================
# 3. Données du graphique
# ==========================================================

time_data = (
    names[names["preusuel"].isin(prenoms_affiches)]
    .groupby(["annais", "preusuel"], as_index=False)["nombre"]
    .sum()
)

# Pourcentages
totaux_annuels = (
    time_data.groupby("annais", as_index=False)["nombre"]
             .sum()
             .rename(columns={"nombre": "total_annuel"})
)

time_data = time_data.merge(
    totaux_annuels,
    on="annais"
)

time_data["pourcentage"] = (
    100 * time_data["nombre"]
    / time_data["total_annuel"]
)

# Flag effet de mode
time_data["effet_mode"] = (
    time_data["preusuel"].isin(prenoms_mode)
)

# ==========================================================
# 4. Contrôles interactifs
# ==========================================================

mode = alt.param(
    name="Mode",
    value="Nombre",
    bind=alt.binding_radio(
        options=["Nombre", "Pourcentage"],
        name="Affichage : "
    )
)

filtre_mode = alt.param(
    name="FiltreMode",
    value="Tous",
    bind=alt.binding_radio(
        options=["Tous", "Effet de mode"],
        name="Type : "
    )
)

prenom = alt.param(
    name="Prenom",
    value="Tous",
    bind=alt.binding_select(
        options=["Tous"] + prenoms_affiches,
        name="Prénom : "
    )
)

# ==========================================================
# 5. Graphique
# ==========================================================

chart = (
    alt.Chart(time_data)

    .add_params(
        mode,
        filtre_mode,
        prenom
    )

    # Filtre prénom
    .transform_filter(
        "(Prenom == 'Tous') || (datum.preusuel == Prenom)"
    )

    # Filtre effet de mode
    .transform_filter(
        "(FiltreMode == 'Tous') || datum.effet_mode"
    )

    # Nombre / %
    .transform_calculate(
        valeur="""
        Mode == 'Nombre'
        ? datum.nombre
        : datum.pourcentage
        """
    )

    .mark_line(point=True)

    .encode(
        x=alt.X(
            "annais:O",
            title="Année"
        ),

        y=alt.Y(
            "valeur:Q",
            title="Naissances / Pourcentage"
        ),

        color=alt.Color(
            "preusuel:N",
            title="Prénom"
        ),

        tooltip=[
            alt.Tooltip("annais:O", title="Année"),
            alt.Tooltip("preusuel:N", title="Prénom"),
            alt.Tooltip("nombre:Q", title="Naissances"),
            alt.Tooltip(
                "pourcentage:Q",
                title="Pourcentage",
                format=".2f"
            )
        ]
    )

    .properties(
        width=1200,
        height=600,
        title="Évolution des prénoms"
    )

    .interactive()
)

chart

Prénoms détectés comme effet de mode :
['PAMELA', 'AMANDA', 'STÉPHANIE', 'SEVERINE', 'LAURYNE', 'MARIELLE', 'WILFRID', 'MANDY', 'NOE', 'BRIGITTE']


alt.Chart(...)